# Memory Probe Experiment — Full Vector vs Magnitude, Memory vs Entropy

Uses **existing trained ResNet-18 + associative memory** (no retraining).

| Phase | Question |
|-------|----------|
| **1** | Do full vectors \(z_1,\ldots,z_4\) beat magnitudes \(\|z_j\|\) for error detection? |
| **3** | Does unified memory \(z_{\text{combined}}\) beat normalized entropy \(\tilde H\)? |

## Workflow

| Run once | Re-run when changing test size |
|----------|--------------------------------|
| **Configuration (paths)** → **Setup** → **Deploy & cache features** | **Eval configuration** → **Phase 1** → **Phase 3** |

Deployment and probe fitting on **full calibration** are cached. Changing `TEST_SAMPLE_SIZE` only subsamples the cached **full test** features and recomputes metrics (seconds, not minutes).

## Configuration — paths & one-time deploy

Run this cell once (or when you need to force a fresh deployment).

In [1]:
from pathlib import Path

TASK = "dermamnist"
SEEDS = (42, 123, 456)
LOGISTIC_C = 1.0  # L2 strength (sklearn: smaller C = stronger reg)

# Set True only to re-run slow deployment inference
FORCE_REDEPLOY = False

REPO_ROOT = Path.cwd().resolve()
if not (REPO_ROOT / "research").exists():
    REPO_ROOT = REPO_ROOT.parent

ARTIFACT_DIR = REPO_ROOT / "research" / "associative_memory_fft" / "artifacts"
DATA_DIR = REPO_ROOT / "data" / "clinical"
OUTPUT_DIR = REPO_ROOT / "research" / "associative_memory_fft" / "probe_experiments"
FEATURE_CACHE_DIR = OUTPUT_DIR / "feature_cache"
FEATURE_CACHE_DIR.mkdir(parents=True, exist_ok=True)

print("REPO_ROOT", REPO_ROOT)
print("FEATURE_CACHE_DIR", FEATURE_CACHE_DIR)
print("FORCE_REDEPLOY", FORCE_REDEPLOY)
print("SEEDS", SEEDS)

REPO_ROOT C:\Users\Sounak Sinha\OneDrive\Desktop\Research\optimization-history-failure-detection
FEATURE_CACHE_DIR C:\Users\Sounak Sinha\OneDrive\Desktop\Research\optimization-history-failure-detection\research\associative_memory_fft\probe_experiments\feature_cache
FORCE_REDEPLOY False
SEEDS (42, 123, 456)


## Setup

Load bundle, deploy \(z_j\) features on cal/test (label-free memory query; labels joined only for \(e\)).

In [2]:
from __future__ import annotations

import json
import pickle
import sys
from typing import Any

import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score, roc_auc_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from research.common.clinical_datasets import ClinicalDatasetConfig, load_clinical_bundle
from research.common.clinical_training import checkpoint_path
from research.common.deployment_pipeline import run_deployment_on_split
from research.common.memory import associative_artifacts_exist, load_associative_artifacts

MAG_COLS = ["z1_magnitude", "z2_magnitude", "z3_magnitude", "z4_magnitude"]
NUM_Z_DIMS = 7


def full_vector_cols() -> list[str]:
    cols: list[str] = []
    for j in range(1, 5):
        cols.extend([f"z{j}_d{d}" for d in range(NUM_Z_DIMS)])
    return cols


FULL_COLS = full_vector_cols()


def feature_cache_paths(seed: int) -> tuple[Path, Path]:
    seed_dir = FEATURE_CACHE_DIR / f"seed{seed}"
    return seed_dir / "cal_features.csv", seed_dir / "test_features_full.csv"


def subsample_test_df(test_full: pd.DataFrame, *, sample_size: int | None, seed: int) -> pd.DataFrame:
    if sample_size is None or sample_size >= len(test_full):
        return test_full.reset_index(drop=True)
    rng = np.random.default_rng(seed)
    idx = np.sort(rng.choice(len(test_full), size=int(sample_size), replace=False))
    return test_full.iloc[idx].reset_index(drop=True)


def fit_l2_probe_on_cal(
    cal_df: pd.DataFrame,
    feature_cols: list[str],
    *,
    representation: str,
) -> dict[str, Any]:
    """Fit scaler + L2 logistic on calibration only."""
    req = feature_cols + ["error"]
    mask_cal = cal_df[req].notna().all(axis=1).to_numpy()
    x_cal = cal_df.loc[mask_cal, feature_cols].to_numpy(dtype=np.float64)
    y_cal = cal_df.loc[mask_cal, "error"].to_numpy(dtype=int)

    if len(x_cal) < 10:
        raise ValueError(f"Too few calibration samples for {representation}: n={len(x_cal)}")
    if len(np.unique(y_cal)) < 2:
        raise ValueError(f"Calibration errors are single-class for {representation}")

    pipe = Pipeline(
        [
            ("scaler", StandardScaler()),
            (
                "clf",
                LogisticRegression(C=LOGISTIC_C, max_iter=5000, random_state=42),
            ),
        ]
    )
    pipe.fit(x_cal, y_cal)

    scaler = pipe.named_steps["scaler"]
    clf = pipe.named_steps["clf"]
    coef_scaled = clf.coef_.reshape(-1)
    coef_raw = coef_scaled / scaler.scale_

    coef_rows = [
        {
            "feature": name,
            "coef_standardized": float(w_s),
            "coef_raw_scale": float(w_r),
        }
        for name, w_s, w_r in zip(feature_cols, coef_scaled, coef_raw, strict=True)
    ]

    return {
        "representation": representation,
        "feature_cols": feature_cols,
        "n_cal": int(len(x_cal)),
        "intercept": float(clf.intercept_[0]),
        "coefficients": pd.DataFrame(coef_rows),
        "pipeline": pipe,
    }


def eval_l2_probe(
    fitted: dict[str, Any],
    test_df: pd.DataFrame,
) -> dict[str, Any]:
    """Evaluate a calibration-fitted probe on a (possibly subsampled) test set."""
    feature_cols = fitted["feature_cols"]
    req = feature_cols + ["error"]
    mask_test = test_df[req].notna().all(axis=1).to_numpy()
    x_test = test_df.loc[mask_test, feature_cols].to_numpy(dtype=np.float64)
    y_test = test_df.loc[mask_test, "error"].to_numpy(dtype=int)

    if len(x_test) < 10:
        raise ValueError(f"Too few test samples: n={len(x_test)}")
    if len(np.unique(y_test)) < 2:
        raise ValueError("Test errors are single-class for this subset")

    pipe = fitted["pipeline"]
    scores = pipe.predict_proba(x_test)[:, 1]
    return {
        **fitted,
        "n_test": int(len(x_test)),
        "auroc": float(roc_auc_score(y_test, scores)),
        "auprc": float(average_precision_score(y_test, scores)),
        "test_scores": scores,
        "test_errors": y_test,
    }


def fit_and_eval_probe(
    cal_df: pd.DataFrame,
    test_df: pd.DataFrame,
    feature_cols: list[str],
    *,
    representation: str,
) -> dict[str, Any]:
    fitted = fit_l2_probe_on_cal(cal_df, feature_cols, representation=representation)
    return eval_l2_probe(fitted, test_df)


def eval_entropy_baseline(test_df: pd.DataFrame) -> dict[str, float]:
    mask = test_df[["normalized_entropy", "error"]].notna().all(axis=1).to_numpy()
    y = test_df.loc[mask, "error"].to_numpy(dtype=int)
    h = test_df.loc[mask, "normalized_entropy"].to_numpy(dtype=float)
    if len(y) < 10 or len(np.unique(y)) < 2:
        return {"n_test": int(len(y)), "auroc": float("nan"), "auprc": float("nan")}
    return {
        "n_test": int(len(y)),
        "auroc": float(roc_auc_score(y, h)),
        "auprc": float(average_precision_score(y, h)),
    }


def deploy_split_for_seed(
    seed: int,
    x: np.ndarray,
    y: np.ndarray,
    ids: np.ndarray,
    *,
    include_history: bool = False,
) -> pd.DataFrame:
    if not associative_artifacts_exist(ARTIFACT_DIR, TASK, seed):
        raise FileNotFoundError(f"Missing artifacts for seed={seed}")
    with open(checkpoint_path(ARTIFACT_DIR, TASK, seed), "rb") as f:
        params = pickle.load(f)["params"]
    mem_state, checkpoints, _ = load_associative_artifacts(ARTIFACT_DIR, TASK, seed)
    _, df = run_deployment_on_split(
        params,
        x,
        y,
        ids,
        mem_state,
        checkpoints,
        num_classes=bundle.num_classes,
        include_history=include_history,
    )
    return df


def load_or_deploy_features(seed: int) -> tuple[pd.DataFrame, pd.DataFrame]:
    cal_path, test_path = feature_cache_paths(seed)
    if not FORCE_REDEPLOY and cal_path.exists() and test_path.exists():
        return pd.read_csv(cal_path), pd.read_csv(test_path)

    print(f"  deploying seed={seed} (full cal + full test) ...", flush=True)
    cal_df = deploy_split_for_seed(
        seed,
        bundle.x_cal,
        bundle.y_cal,
        bundle.sample_ids["cal"],
    )
    test_full = deploy_split_for_seed(
        seed,
        bundle.x_test,
        bundle.y_test,
        bundle.sample_ids["test"],
    )
    cal_path.parent.mkdir(parents=True, exist_ok=True)
    cal_df.to_csv(cal_path, index=False)
    test_full.to_csv(test_path, index=False)
    return cal_df, test_full


bundle = load_clinical_bundle(ClinicalDatasetConfig(task=TASK, data_dir=DATA_DIR))
print(f"bundle: n_cal={len(bundle.x_cal)} n_test={len(bundle.x_test)} num_classes={bundle.num_classes}")

bundle: n_cal=1602 n_test=2005 num_classes=7


In [3]:
# One-time: deploy full cal + full test features (cached to FEATURE_CACHE_DIR)

cal_frames_full: dict[int, pd.DataFrame] = {}
test_frames_full: dict[int, pd.DataFrame] = {}

for seed in SEEDS:
    cal_df, test_full = load_or_deploy_features(seed)
    cal_frames_full[seed] = cal_df
    test_frames_full[seed] = test_full
    print(f"seed={seed}: cal={len(cal_df)} test_full={len(test_full)} cached={not FORCE_REDEPLOY}")

print("feature cache ready →", FEATURE_CACHE_DIR)

seed=42: cal=1602 test_full=2005 cached=True
seed=123: cal=1602 test_full=2005 cached=True
seed=456: cal=1602 test_full=2005 cached=True
feature cache ready → C:\Users\Sounak Sinha\OneDrive\Desktop\Research\optimization-history-failure-detection\research\associative_memory_fft\probe_experiments\feature_cache


## Eval configuration — change test size here

Re-run **this cell** and the **Phase 1 / Phase 3** cells below. No redeployment.

In [4]:
TEST_SAMPLE_SIZE = 2000  # None = full test set
RANDOM_SEED = 42

# Build eval subsets from cached full test features
cal_frames = cal_frames_full
test_frames = {
    seed: subsample_test_df(test_frames_full[seed], sample_size=TEST_SAMPLE_SIZE, seed=RANDOM_SEED)
    for seed in SEEDS
}

print(f"TEST_SAMPLE_SIZE={TEST_SAMPLE_SIZE}  RANDOM_SEED={RANDOM_SEED}")
for seed in SEEDS:
    te = test_frames[seed]
    print(f"  seed={seed}: n_test={len(te)} error_rate={te['error'].mean():.3f}")

TEST_SAMPLE_SIZE=2000  RANDOM_SEED=42
  seed=42: n_test=2000 error_rate=0.332
  seed=123: n_test=2000 error_rate=0.332
  seed=456: n_test=2000 error_rate=0.332


---

# Phase 1 — \(\|z\|\) vs Full \(z\)

\[
X_{\text{mag}}=[s_1,s_2,s_3,s_4],\quad
X_{\text{full}}=[z_1,z_2,z_3,z_4]\in\mathbb R^{28}
\]

Train separate L2 logistic probes on calibration; evaluate on the held-out test subset.
Select \(X_{\text{best}}\) by **calibration AUROC** (mean across seeds). Test AUROC is reported only for final evaluation.

In [5]:
phase1_rows: list[dict[str, Any]] = []
phase1_coefs: dict[int, dict[str, pd.DataFrame]] = {}
phase1_runs: dict[int, dict[str, Any]] = {}

for seed in SEEDS:
    cal_df = cal_frames[seed]
    test_df = test_frames[seed]
    mag_fit = fit_l2_probe_on_cal(cal_df, MAG_COLS, representation="magnitude")
    full_fit = fit_l2_probe_on_cal(cal_df, FULL_COLS, representation="full_vector")
    mag_cal = eval_l2_probe(mag_fit, cal_df)
    full_cal = eval_l2_probe(full_fit, cal_df)
    mag = eval_l2_probe(mag_fit, test_df)
    full = eval_l2_probe(full_fit, test_df)
    phase1_runs[seed] = {"magnitude": mag, "full_vector": full}
    phase1_coefs[seed] = {
        "magnitude": mag["coefficients"],
        "full_vector": full["coefficients"],
    }
    for run, cal_run in ((mag, mag_cal), (full, full_cal)):
        phase1_rows.append(
            {
                "seed": seed,
                "representation": run["representation"],
                "n_cal": run["n_cal"],
                "n_test": run["n_test"],
                "cal_auroc": cal_run["auroc"],
                "test_auroc": run["auroc"],
                "test_auprc": run["auprc"],
                "intercept": run["intercept"],
            }
        )

phase1_df = pd.DataFrame(phase1_rows)
phase1_summary = (
    phase1_df.groupby("representation")[["cal_auroc", "test_auroc", "test_auprc"]]
    .agg(["mean", "std", "min", "max"])
    .round(4)
)

mag_cal_mean = float(phase1_df.loc[phase1_df["representation"] == "magnitude", "cal_auroc"].mean())
full_cal_mean = float(phase1_df.loc[phase1_df["representation"] == "full_vector", "cal_auroc"].mean())
mag_test_mean = float(phase1_df.loc[phase1_df["representation"] == "magnitude", "test_auroc"].mean())
full_test_mean = float(phase1_df.loc[phase1_df["representation"] == "full_vector", "test_auroc"].mean())
if full_cal_mean >= mag_cal_mean:
    x_best_name = "full_vector"
    x_best_cols = FULL_COLS
else:
    x_best_name = "magnitude"
    x_best_cols = MAG_COLS

phase1_decision = {
    "x_best": x_best_name,
    "mean_cal_auroc_magnitude": mag_cal_mean,
    "mean_cal_auroc_full_vector": full_cal_mean,
    "mean_test_auroc_magnitude": mag_test_mean,
    "mean_test_auroc_full_vector": full_test_mean,
    "full_beats_magnitude_on_cal_auroc": full_cal_mean >= mag_cal_mean,
    "feature_dim": len(x_best_cols),
}

print("Phase 1 per-seed results")
display(phase1_df)
print("\nPhase 1 aggregate (mean ± std across seeds)")
display(phase1_summary)
print("\nPhase 1 decision:", phase1_decision)

for seed in SEEDS:
    print(f"\n--- seed={seed} magnitude coefficients ---")
    display(phase1_coefs[seed]["magnitude"])
    print(f"--- seed={seed} full-vector coefficients (first 10) ---")
    display(phase1_coefs[seed]["full_vector"].head(10))

Phase 1 per-seed results


,seed,representation,n_cal,n_test,cal_auroc,test_auroc,test_auprc,intercept
0,42,magnitude,1602,2000,0.681465,0.674985,0.530704,-0.775251
1,42,full_vector,1602,2000,0.765180,0.733530,0.583675,-0.845694
2,123,magnitude,1602,2000,0.693032,0.677705,0.489806,-0.804832
3,123,full_vector,1602,2000,0.762137,0.731191,0.572207,-0.871757
4,456,magnitude,1602,2000,0.656356,0.675040,0.525522,-0.752532
5,456,full_vector,1602,2000,0.762722,0.737709,0.597915,-0.821014



Phase 1 aggregate (mean ± std across seeds)


cal_auroc                         test_auroc                  \
                    mean     std     min     max       mean     std     min   
representation                                                                
full_vector       0.7633  0.0016  0.7621  0.7652     0.7341  0.0033  0.7312   
magnitude         0.6770  0.0188  0.6564  0.6930     0.6759  0.0016  0.6750   

                       test_auprc                          
                   max       mean     std     min     max  
representation                                             
full_vector     0.7377     0.5846  0.0129  0.5722  0.5979  
magnitude       0.6777     0.5153  0.0223  0.4898  0.5307


Phase 1 decision: {'x_best': 'full_vector', 'mean_cal_auroc_magnitude': 0.6769511836326255, 'mean_cal_auroc_full_vector': 0.7633462939505571, 'mean_test_auroc_magnitude': 0.6759101009178003, 'mean_test_auroc_full_vector': 0.7341432478482063, 'full_beats_magnitude_on_cal_auroc': True, 'feature_dim': 28}

--- seed=42 magnitude coefficients ---


...

,feature,coef_standardized,coef_raw_scale
0,z1_d0,0.082503,201.619898
1,z1_d1,0.270638,558.895526
2,z1_d2,0.827255,1061.030081
3,z1_d3,1.288458,5018.088891
4,z1_d4,-1.092963,-1198.211378
5,z1_d5,-0.055693,-59.576884
6,z1_d6,-0.171314,-320.507689
7,z2_d0,-0.439980,-2478.209462
8,z2_d1,-0.998571,-4689.644372
9,z2_d2,-0.664724,-1840.940832



--- seed=456 magnitude coefficients ---


,feature,coef_standardized,coef_raw_scale
0,z1_magnitude,-0.377892,-588.794982
1,z2_magnitude,0.491580,2265.349464
2,z3_magnitude,-0.264311,-20371.652996
3,z4_magnitude,0.278844,37745.246128


--- seed=456 full-vector coefficients (first 10) ---


,feature,coef_standardized,coef_raw_scale
0,z1_d0,0.333455,892.797796
1,z1_d1,-0.462797,-1170.958462
2,z1_d2,-0.018020,-28.468318
3,z1_d3,-0.255650,-2019.131463
4,z1_d4,-0.082839,-183.286086
5,z1_d5,-0.155885,-211.651515
6,z1_d6,0.891961,3127.161264
7,z2_d0,-1.192569,-8174.539436
8,z2_d1,-0.880206,-3478.817966
9,z2_d2,0.641646,1811.801990


---

# Phase 3 — Unified Memory vs Entropy

Using \(X_{\text{best}}\) from Phase 1, train one probe on calibration:

\[
z_{\text{combined}}(x)=P(e=1\mid X_{\text{best}})
\]

Compare against normalized entropy baseline \(\tilde H(x)\) on the same test subset.

In [6]:
phase3_rows: list[dict[str, Any]] = []
phase3_coefs: dict[int, pd.DataFrame] = {}

for seed in SEEDS:
    cal_df = cal_frames[seed]
    test_df = test_frames[seed]
    mem = fit_and_eval_probe(cal_df, test_df, x_best_cols, representation=f"z_combined_{x_best_name}")
    ent = eval_entropy_baseline(test_df)
    phase3_coefs[seed] = mem["coefficients"]

    delta_auroc = mem["auroc"] - ent["auroc"]
    delta_auprc = mem["auprc"] - ent["auprc"]

    for model, metrics in [
        ("entropy_H", ent),
        ("z_combined", {"auroc": mem["auroc"], "auprc": mem["auprc"], "n_test": mem["n_test"]}),
    ]:
        phase3_rows.append(
            {
                "seed": seed,
                "model": model,
                "representation": x_best_name,
                "n_cal": mem["n_cal"],
                "n_test": metrics["n_test"],
                "auroc": metrics["auroc"],
                "auprc": metrics["auprc"],
            }
        )

    phase3_rows.append(
        {
            "seed": seed,
            "model": "delta_z_combined_minus_H",
            "representation": x_best_name,
            "n_cal": mem["n_cal"],
            "n_test": mem["n_test"],
            "auroc": delta_auroc,
            "auprc": delta_auprc,
        }
    )

phase3_df = pd.DataFrame(phase3_rows)
pivot = phase3_df.pivot_table(index="seed", columns="model", values=["auroc", "auprc"], aggfunc="first")

z_auroc_mean = float(phase3_df.loc[phase3_df["model"] == "z_combined", "auroc"].mean())
h_auroc_mean = float(phase3_df.loc[phase3_df["model"] == "entropy_H", "auroc"].mean())
z_auprc_mean = float(phase3_df.loc[phase3_df["model"] == "z_combined", "auprc"].mean())
h_auprc_mean = float(phase3_df.loc[phase3_df["model"] == "entropy_H", "auprc"].mean())

phase3_summary = {
    "x_best": x_best_name,
    "mean_auroc_z_combined": z_auroc_mean,
    "mean_auroc_H": h_auroc_mean,
    "mean_delta_auroc": z_auroc_mean - h_auroc_mean,
    "mean_auprc_z_combined": z_auprc_mean,
    "mean_auprc_H": h_auprc_mean,
    "mean_delta_auprc": z_auprc_mean - h_auprc_mean,
    "z_combined_beats_H_on_mean_auroc": z_auroc_mean > h_auroc_mean,
    "test_sample_size": int(len(test_frames[SEEDS[0]])),
    "logistic_C": LOGISTIC_C,
}

results_payload = {
    "config": {
        "task": TASK,
        "seeds": list(SEEDS),
        "test_sample_size": TEST_SAMPLE_SIZE,
        "random_seed": RANDOM_SEED,
        "logistic_C": LOGISTIC_C,
    },
    "phase1_decision": phase1_decision,
    "phase1_per_seed": phase1_df.to_dict(orient="records"),
    "phase3_summary": phase3_summary,
    "phase3_per_seed": phase3_df.to_dict(orient="records"),
}
with open(OUTPUT_DIR / "probe_results.json", "w", encoding="utf-8") as f:
    json.dump(results_payload, f, indent=2, default=str)

phase1_df.to_csv(OUTPUT_DIR / "phase1_per_seed.csv", index=False)
phase3_df.to_csv(OUTPUT_DIR / "phase3_per_seed.csv", index=False)

print("Phase 3 per-seed comparison")
display(phase3_df)
print("\nPhase 3 pivot (AUROC / AUPRC by seed)")
display(pivot)
print("\nPhase 3 summary:")
for k, v in phase3_summary.items():
    print(f"  {k}: {v}")

for seed in SEEDS:
    print(f"\n--- seed={seed} z_combined coefficients ({x_best_name}) ---")
    display(phase3_coefs[seed])

Phase 3 per-seed comparison


,seed,model,representation,n_cal,n_test,auroc,auprc
0,42,entropy_H,full_vector,1602,2000,0.674489,0.477077
1,42,z_combined,full_vector,1602,2000,0.733530,0.583675
2,42,delta_z_combined_minus_H,full_vector,1602,2000,0.059041,0.106599
3,123,entropy_H,full_vector,1602,2000,0.672417,0.498044
4,123,z_combined,full_vector,1602,2000,0.731191,0.572207
5,123,delta_z_combined_minus_H,full_vector,1602,2000,0.058774,0.074163
6,456,entropy_H,full_vector,1602,2000,0.587957,0.406972
7,456,z_combined,full_vector,1602,2000,0.737709,0.597915
8,456,delta_z_combined_minus_H,full_vector,1602,2000,0.149752,0.190943



Phase 3 pivot (AUROC / AUPRC by seed)


auprc                                         auroc  \
model delta_z_combined_minus_H entropy_H z_combined delta_z_combined_minus_H   
seed                                                                           
42                    0.106599  0.477077   0.583675                 0.059041   
123                   0.074163  0.498044   0.572207                 0.058774   
456                   0.190943  0.406972   0.597915                 0.149752   

                            
model entropy_H z_combined  
seed                        
42     0.674489   0.733530  
123    0.672417   0.731191  
456    0.587957   0.737709


Phase 3 summary:
  x_best: full_vector
  mean_auroc_z_combined: 0.7341432478482063
  mean_auroc_H: 0.6449541287853576
  mean_delta_auroc: 0.08918911906284865
  mean_auprc_z_combined: 0.5845989469650507
  mean_auprc_H: 0.4606974898917427
  mean_delta_auprc: 0.12390145707330796
  z_combined_beats_H_on_mean_auroc: True
  test_sample_size: 2000
  logistic_C: 1.0

--- seed=42 z_combined coefficients (full_vector) ---


,feature,coef_standardized,coef_raw_scale
0,z1_d0,0.357883,997.484962
1,z1_d1,0.671355,1637.176060
2,z1_d2,0.154558,195.941939
3,z1_d3,0.567083,3482.076155
4,z1_d4,-1.164196,-2421.867081
5,z1_d5,-0.360367,-525.871679
6,z1_d6,0.357037,676.025105
7,z2_d0,-0.062713,-773.472832
8,z2_d1,-1.181068,-7785.628997
9,z2_d2,-1.180809,-4780.671805



--- seed=123 z_combined coefficients (full_vector) ---


,feature,coef_standardized,coef_raw_scale
0,z1_d0,0.082503,201.619898
1,z1_d1,0.270638,558.895526
2,z1_d2,0.827255,1061.030081
3,z1_d3,1.288458,5018.088891
4,z1_d4,-1.092963,-1198.211378
5,z1_d5,-0.055693,-59.576884
6,z1_d6,-0.171314,-320.507689
7,z2_d0,-0.439980,-2478.209462
8,z2_d1,-0.998571,-4689.644372
9,z2_d2,-0.664724,-1840.940832



--- seed=456 z_combined coefficients (full_vector) ---


,feature,coef_standardized,coef_raw_scale
0,z1_d0,0.333455,892.797796
1,z1_d1,-0.462797,-1170.958462
2,z1_d2,-0.018020,-28.468318
3,z1_d3,-0.255650,-2019.131463
4,z1_d4,-0.082839,-183.286086
5,z1_d5,-0.155885,-211.651515
6,z1_d6,0.891961,3127.161264
7,z2_d0,-1.192569,-8174.539436
8,z2_d1,-0.880206,-3478.817966
9,z2_d2,0.641646,1811.801990
